<a href="https://colab.research.google.com/github/shrutimali25/CareFlow/blob/feature%2Fcareflow-patient-analysis/notebooks/CareFlow_Process_Mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CareFlow — Clinical Pathway Process Mining

## Purpose
Analyze hospital event logs to understand patient journeys, discover process flows, and investigate elapsed times between activities.

## Analysis Plan
1. Inspect and validate the source data.
2. Understand the event-log structure.
3. Discover process maps using PM4Py.
4. Analyze common pathways and repeated activities.
5. Calculate elapsed times between activities.
6. Prepare validated outputs for Power BI.
7. Document findings and limitations.

## Data Integrity
All reported results must be calculated from the actual source data and validated before reporting.

In [ ]:
import sys
from datetime import datetime

print("CareFlow environment initialized")
print("Python version:", sys.version.split()[0])
print("Notebook started:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

CareFlow environment initialized
Python version: 3.13.16
Notebook started: 2026-10-22 13:44:15


In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nFiles uploaded:")
for filename in uploaded:
    print("-", filename)

Saving patient_case_summary.csv to patient_case_summary.csv
Saving activity_frequency.csv to activity_frequency.csv
Saving activity_time_differences.csv to activity_time_differences.csv

Files uploaded:
- patient_case_summary.csv
- activity_frequency.csv
- activity_time_differences.csv


In [ ]:
import pandas as pd
from pathlib import Path

# CareFlow: load the three analysis files
data_files = {
    "patient_cases": "patient_case_summary.csv",
    "activity_frequency": "activity_frequency.csv",
    "activity_times": "activity_time_differences.csv"
}

data = {}

for name, filename in data_files.items():
    path = Path(filename)

    if not path.exists():
        print(f"File not found: {filename}")
        continue

    data[name] = pd.read_csv(path)
    df = data[name]

    print(f"\n{'=' * 55}")
    print(f"FILE: {filename}")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print("Column names:", df.columns.tolist())
    print("\nMissing values per column:")
    print(df.isna().sum().to_string())

print("\nCareFlow file inspection completed.")


FILE: patient_case_summary.csv
Rows: 1049
Columns: 5
Column names: ['case:concept:name', 'total_events', 'first_event_time', 'last_event_time', 'treatment_duration_hours']

Missing values per column:
case:concept:name           0
total_events                0
first_event_time            0
last_event_time             0
treatment_duration_hours    0

FILE: activity_frequency.csv
Rows: 16
Columns: 3
Column names: ['concept:name', 'total_occurrences', 'unique_cases']

Missing values per column:
concept:name         0
total_occurrences    0
unique_cases         0

FILE: activity_time_differences.csv
Rows: 115
Columns: 5
Column names: ['previous_activity', 'concept:name', 'total_transitions', 'average_minutes', 'median_minutes']

Missing values per column:
previous_activity    0
concept:name         0
total_transitions    0
average_minutes      0
median_minutes       0

CareFlow file inspection completed.


In [ ]:

import pandas as pd
from pathlib import Path

data_files = {
    "patient_cases": "patient_case_summary.csv",
    "activity_frequency": "activity_frequency.csv",
    "activity_times": "activity_time_differences.csv"
}

data = {}

for name, filename in data_files.items():
    if not Path(filename).exists():
        raise FileNotFoundError(
            f"{filename} is missing. Please upload the CSV files again."
        )
    data[name] = pd.read_csv(filename)

print("All three CareFlow CSV files loaded successfully.")

for name, df in data.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

All three CareFlow CSV files loaded successfully.
patient_cases: 1049 rows, 5 columns
activity_frequency: 16 rows, 3 columns
activity_times: 115 rows, 5 columns


In [ ]:
# CareFlow: Data Quality and Consistency Validation

cases = data["patient_cases"].copy()
frequency = data["activity_frequency"].copy()
transitions = data["activity_times"].copy()

# 1. Validate patient case IDs
print("1. CASE ID VALIDATION")
print("Duplicate case IDs:",
      cases["case:concept:name"].duplicated().sum())
print("Unique case IDs:",
      cases["case:concept:name"].nunique())

# 2. Compare event totals
case_event_total = pd.to_numeric(
    cases["total_events"], errors="coerce"
).sum()

activity_event_total = pd.to_numeric(
    frequency["total_occurrences"], errors="coerce"
).sum()

print("\n2. EVENT COUNT VALIDATION")
print("Events in case summary:", int(case_event_total))
print("Events in activity frequency:", int(activity_event_total))
print("Event totals match:", case_event_total == activity_event_total)

# 3. Validate activity counts
print("\n3. ACTIVITY FREQUENCY VALIDATION")
print("Distinct activities:", frequency["concept:name"].nunique())

print("Negative activity occurrence counts:",
      (pd.to_numeric(frequency["total_occurrences"],
                     errors="coerce") < 0).sum())

print("Activities with unique-case counts exceeding total cases:",
      (pd.to_numeric(frequency["unique_cases"],
                     errors="coerce")
       > cases["case:concept:name"].nunique()).sum())

# 4. Validate timestamps and visit durations
print("\n4. TIMESTAMP AND DURATION VALIDATION")

cases["first_event_time"] = pd.to_datetime(
    cases["first_event_time"], errors="coerce", utc=True
)
cases["last_event_time"] = pd.to_datetime(
    cases["last_event_time"], errors="coerce", utc=True
)
cases["treatment_duration_hours"] = pd.to_numeric(
    cases["treatment_duration_hours"], errors="coerce"
)

invalid_timestamps = (
    cases["first_event_time"].isna()
    | cases["last_event_time"].isna()
)

negative_elapsed = (
    cases["last_event_time"] < cases["first_event_time"]
).fillna(False)

calculated_hours = (
    cases["last_event_time"] - cases["first_event_time"]
).dt.total_seconds() / 3600

duration_difference = (
    calculated_hours - cases["treatment_duration_hours"]
).abs()

print("Invalid timestamps:", int(invalid_timestamps.sum()))
print("Last event before first event:", int(negative_elapsed.sum()))
print("Missing or invalid durations:",
      int(cases["treatment_duration_hours"].isna().sum()))
print("Negative durations:",
      int((cases["treatment_duration_hours"] < 0).sum()))
print("Durations differing from timestamps by over 1 minute:",
      int((duration_difference > 1/60).sum()))

# 5. Validate transition data
print("\n5. TRANSITION VALIDATION")

transition_counts = pd.to_numeric(
    transitions["total_transitions"], errors="coerce"
)
average_minutes = pd.to_numeric(
    transitions["average_minutes"], errors="coerce"
)
median_minutes = pd.to_numeric(
    transitions["median_minutes"], errors="coerce"
)

print("Transition pairs:", len(transitions))
print("Invalid or non-positive transition counts:",
      int(((transition_counts <= 0) |
           transition_counts.isna()).sum()))
print("Negative average times:",
      int((average_minutes < 0).sum()))
print("Negative median times:",
      int((median_minutes < 0).sum()))
print("Rows with missing transition metrics:",
      int(transitions[
          ["total_transitions", "average_minutes", "median_minutes"]
      ].isna().any(axis=1).sum()))

print("\nCareFlow validation checks completed.")

1. CASE ID VALIDATION
Duplicate case IDs: 0
Unique case IDs: 1049

2. EVENT COUNT VALIDATION
Events in case summary: 15189
Events in activity frequency: 15189
Event totals match: True

3. ACTIVITY FREQUENCY VALIDATION
Distinct activities: 16
Negative activity occurrence counts: 0
Activities with unique-case counts exceeding total cases: 0

4. TIMESTAMP AND DURATION VALIDATION
Invalid timestamps: 0
Last event before first event: 0
Missing or invalid durations: 0
Negative durations: 0
Durations differing from timestamps by over 1 minute: 0

5. TRANSITION VALIDATION
Transition pairs: 115
Invalid or non-positive transition counts: 0
Negative average times: 0
Negative median times: 0
Rows with missing transition metrics: 0

CareFlow validation checks completed.


In [ ]:
# CareFlow: Patient Pathway and Bottleneck Analysis

import pandas as pd

frequency = data["activity_frequency"].copy()
transitions = data["activity_times"].copy()
cases = data["patient_cases"].copy()

# Ensure numeric columns are numeric
frequency["total_occurrences"] = pd.to_numeric(
    frequency["total_occurrences"], errors="coerce"
)
frequency["unique_cases"] = pd.to_numeric(
    frequency["unique_cases"], errors="coerce"
)

for col in ["total_transitions", "average_minutes", "median_minutes"]:
    transitions[col] = pd.to_numeric(
        transitions[col], errors="coerce"
    )

# 1. Most frequent activities
print("=" * 65)
print("1. TOP 10 MOST FREQUENT ACTIVITIES")
print("=" * 65)

top_activities = frequency.sort_values(
    "total_occurrences", ascending=False
).head(10)

display(top_activities)

# 2. Most frequent activity-to-activity transitions
print("\n" + "=" * 65)
print("2. TOP 10 MOST FREQUENT TRANSITIONS")
print("=" * 65)

top_transitions = transitions.sort_values(
    "total_transitions", ascending=False
).head(10)

display(top_transitions[
    ["previous_activity", "concept:name",
     "total_transitions", "average_minutes", "median_minutes"]
])

# 3. Transitions with the longest average time gaps
print("\n" + "=" * 65)
print("3. TOP 10 LONGEST AVERAGE TRANSITION TIMES")
print("=" * 65)

longest_average = transitions.sort_values(
    "average_minutes", ascending=False
).head(10)

display(longest_average[
    ["previous_activity", "concept:name",
     "total_transitions", "average_minutes", "median_minutes"]
])

# 4. Transitions with the longest median time gaps
print("\n" + "=" * 65)
print("4. TOP 10 LONGEST MEDIAN TRANSITION TIMES")
print("=" * 65)

longest_median = transitions.sort_values(
    "median_minutes", ascending=False
).head(10)

display(longest_median[
    ["previous_activity", "concept:name",
     "total_transitions", "average_minutes", "median_minutes"]
])

# 5. Summary of visit durations
print("\n" + "=" * 65)
print("5. PATIENT VISIT DURATION SUMMARY")
print("=" * 65)

cases["treatment_duration_hours"] = pd.to_numeric(
    cases["treatment_duration_hours"], errors="coerce"
)

display(cases["treatment_duration_hours"].describe().to_frame(
    name="Duration (hours)"
))

print("\nPathway and bottleneck exploration completed.")

In [ ]:
# CareFlow: Investigate unusually long time gaps

# 1. All transitions leading to Return ER
print("1. TRANSITIONS LEADING TO RETURN ER")

return_er = transitions[
    transitions["concept:name"].astype(str).str.strip().eq("Return ER")
].copy()

return_er["average_days"] = return_er["average_minutes"] / 1440
return_er["median_days"] = return_er["median_minutes"] / 1440

display(
    return_er[
        ["previous_activity", "concept:name", "total_transitions",
         "average_days", "median_days"]
    ].sort_values("total_transitions", ascending=False)
)

# 2. Longest recorded cases
print("\n2. TEN LONGEST RECORDED CASE DURATIONS")

cases["treatment_duration_hours"] = pd.to_numeric(
    cases["treatment_duration_hours"], errors="coerce"
)

long_cases = cases.nlargest(10, "treatment_duration_hours").copy()
long_cases["duration_days"] = (
    long_cases["treatment_duration_hours"] / 24
)

display(
    long_cases[
        ["case:concept:name", "first_event_time", "last_event_time",
         "treatment_duration_hours", "duration_days"]
    ]
)

# 3. Duration percentiles for context
print("\n3. DURATION PERCENTILES (DAYS)")

print(
    (cases["treatment_duration_hours"] / 24)
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
    .rename("duration_days")
)

Milestone 2 & 3 — Data Validation and Initial Analysis

Data quality validation

- Patient cases analyzed: 1,049
- Total events: 15,189
- Distinct activities: 16
- Activity-to-activity transition pairs: 115
- Duplicate case IDs: 0
- Event totals across summary files: matched
- Invalid timestamps and negative durations: 0
- Duration mismatches exceeding one minute: 0

Initial pathway findings

- Leucocytes and CRP were the most frequent activities.
- ER Registration → ER Triage occurred 970 times, with an average transition gap of approximately 10.6 minutes.
- ER Triage → ER Sepsis Triage occurred 904 times, with an average transition gap of approximately 2.9 minutes.
- Some transitions to Return ER showed long gaps and require interpretation as possible repeat visits rather than confirmed hospital bottlenecks.
- Recorded case durations were strongly right-skewed, so average duration should be interpreted alongside the median.

Next steps

- Validate the original XES event log against the summary CSV files.
- Generate a process map using PM4Py.
- Analyze common pathways, repeated activities, and potential bottlenecks.
- Prepare validated aggregate metrics for the Power BI dashboard.

Note: Findings are preliminary and based on the supplied CSV summaries. Raw event-log validation is pending.

In [ ]:
from google.colab import files

uploaded_xes = files.upload()

Saving sepsis.xes.gz to sepsis.xes.gz


In [ ]:
from pathlib import Path

for file in Path("/content").glob("sepsis*"):
    print("File found:", file.name)
    print("File size:", round(file.stat().st_size / 1024, 2), "KB")

File found: sepsis.xes.gz
File size: 197.76 KB


In [ ]:
%pip install pm4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 68.2 MB/s eta 0:00:00


In [ ]:
import pm4py

print("PM4Py installed successfully!")
print("PM4Py version:", pm4py.__version__)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




PM4Py installed successfully!
PM4Py version: 2.7.23.11
